# Topic 4: Testing PySpark Code

> **Phase 5 → Week 9 → Topic 4**

---

## Why Testing Spark is Different

```
Unit testing Spark has challenges:

  1. SparkSession is heavy (JVM startup, memory allocation)
     → Use a shared session-scoped fixture (not per-test)

  2. DataFrames are lazy — no data until an action
     → Tests must call .collect(), .count(), .show() to materialize

  3. DataFrame equality is tricky (column order, row order, nulls)
     → Use helper functions that handle these cases

  4. Side effects (writes, JDBC, external calls) are hard to test
     → Separate transformation logic from I/O; test transformations only

  5. Non-determinism (rand(), uuid(), current_timestamp())
     → Seed random functions or abstract them behind interfaces
```

---

## Interview Cheat Sheet

**Q: How do you write unit tests for a PySpark transformation function?**
> Separate I/O from transformation logic. Write pure transformation functions that take a DataFrame and return a DataFrame. In tests: (1) create a small in-memory DataFrame with `spark.createDataFrame([...])`, (2) call the transformation, (3) assert the output using `.collect()` or column comparisons. Use `pytest` with a session-scoped `SparkSession` fixture to avoid JVM startup on every test.

**Q: How do you compare two DataFrames for equality in tests?**
> `df1.collect() == df2.collect()` fails if row order differs. Better: (1) sort both by a stable key, then compare. (2) Use `df1.subtract(df2).count() == 0 and df2.subtract(df1).count() == 0` for unordered equality. (3) Use `chispa` library (`assert_df_equality`) which handles column ordering, null handling, and row ordering automatically.

**Q: What is the difference between unit, integration, and end-to-end tests for ETL?**
> **Unit**: test one transformation function with small in-memory DataFrames — fast (<1s each). **Integration**: test the pipeline reading from and writing to real storage (local Parquet, Delta), with realistic data volumes — slower (10s–2min). **End-to-end**: run the full pipeline against a staging environment with production-like data — slowest, run in CI/CD on merge.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# In a real test file this would be a pytest fixture
spark = SparkSession.builder \
    .appName("Week9 - Testing PySpark") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Structuring Code for Testability

In [ ]:
# ─────────────────────────────────────────────────────────────────
# BAD: I/O and transformation mixed → untestable
# ─────────────────────────────────────────────────────────────────
def bad_etl_pipeline():
    df = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
    df = df.filter(F.col("amount") > 0)
    df = df.withColumn("tier",
        F.when(F.col("amount") > 500, "High")
         .when(F.col("amount") > 100, "Medium")
         .otherwise("Low"))
    df.write.mode("overwrite").parquet("/tmp/output")
    # Can't test the transformation without reading/writing files!

# ─────────────────────────────────────────────────────────────────
# GOOD: pure transformation functions, I/O at the edges
# ─────────────────────────────────────────────────────────────────

# These functions take DataFrames, return DataFrames — easily testable
def filter_valid_orders(df):
    """Remove orders with null or negative amounts."""
    return df.filter(F.col("amount").isNotNull() & (F.col("amount") > 0))

def add_order_tier(df):
    """Add tier column: High/Medium/Low based on amount."""
    return df.withColumn("tier",
        F.when(F.col("amount") > 500, "High")
         .when(F.col("amount") > 100, "Medium")
         .otherwise("Low")
    )

def enrich_with_category(orders_df, products_df):
    """Join orders with products to add category."""
    return orders_df.join(
        F.broadcast(products_df.select("product_id", "category")),
        on="product_id",
        how="left"
    )

# I/O only at the very edges (read in main, write at end)
def run_orders_pipeline(orders_df, products_df):
    """Full pipeline — composing pure functions."""
    return (
        orders_df
        | filter_valid_orders
        # Python doesn't have |> but we can chain:
    )  # shown for clarity — see next cell for chain

print("Transformation functions defined — ready to test")

## Part 2: Writing Unit Tests

In [ ]:
# DataFrame equality helper
def assert_df_equal(actual, expected, sort_cols=None, check_row_order=False):
    """
    Assert two DataFrames are equal.
    sort_cols: columns to sort by before comparing (for order-independent check)
    """
    # Check column sets
    actual_cols = set(actual.columns)
    expected_cols = set(expected.columns)
    assert actual_cols == expected_cols, \
        f"Column mismatch. Extra: {actual_cols - expected_cols}, Missing: {expected_cols - actual_cols}"

    # Reorder to same column order
    actual_ordered   = actual.select(sorted(actual.columns))
    expected_ordered = expected.select(sorted(expected.columns))

    if sort_cols:
        actual_ordered   = actual_ordered.orderBy(sort_cols)
        expected_ordered = expected_ordered.orderBy(sort_cols)

    # Compare
    actual_rows   = actual_ordered.collect()
    expected_rows = expected_ordered.collect()

    if not check_row_order:
        actual_rows   = sorted(actual_rows,   key=str)
        expected_rows = sorted(expected_rows, key=str)

    assert actual_rows == expected_rows, \
        f"Data mismatch.\nActual:   {actual_rows}\nExpected: {expected_rows}"

print("assert_df_equal helper defined")

In [ ]:
# Unit tests for our transformation functions

def test_filter_valid_orders_removes_nulls():
    input_df = spark.createDataFrame([
        ("O001", 250.0), ("O002", None),  ("O003", -50.0), ("O004", 100.0),
    ], StructType([StructField("order_id", StringType(), True),
                   StructField("amount", DoubleType(), True)]))

    expected = spark.createDataFrame([
        ("O001", 250.0), ("O004", 100.0),
    ], ["order_id", "amount"])

    result = filter_valid_orders(input_df)
    assert_df_equal(result, expected, sort_cols=["order_id"])
    print("✓ test_filter_valid_orders_removes_nulls")


def test_add_order_tier_correct_buckets():
    input_df = spark.createDataFrame([
        ("O001",  50.0),
        ("O002", 150.0),
        ("O003", 600.0),
        ("O004", 100.0),  # boundary: <= 100 → Low
        ("O005", 101.0),  # boundary: > 100 → Medium
        ("O006", 500.0),  # boundary: <= 500 → Medium
        ("O007", 501.0),  # boundary: > 500 → High
    ], ["order_id", "amount"])

    result = add_order_tier(input_df).select("order_id", "tier")

    expected = spark.createDataFrame([
        ("O001", "Low"), ("O002", "Medium"), ("O003", "High"),
        ("O004", "Low"), ("O005", "Medium"), ("O006", "Medium"), ("O007", "High"),
    ], ["order_id", "tier"])

    assert_df_equal(result, expected, sort_cols=["order_id"])
    print("✓ test_add_order_tier_correct_buckets")


def test_filter_valid_orders_empty_input():
    empty = spark.createDataFrame([], StructType([
        StructField("order_id", StringType(), True),
        StructField("amount", DoubleType(), True),
    ]))
    result = filter_valid_orders(empty)
    assert result.count() == 0
    print("✓ test_filter_valid_orders_empty_input")


# Run tests
print("Running unit tests...")
test_filter_valid_orders_removes_nulls()
test_add_order_tier_correct_buckets()
test_filter_valid_orders_empty_input()
print("\nAll tests passed!")

## Part 3: pytest Setup for PySpark

In [ ]:
# How to set up pytest for PySpark — write this as conftest.py in your project
print("""
# conftest.py — shared pytest fixtures
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")   # one SparkSession for ALL tests in the session
def spark():
    session = SparkSession.builder \\
        .appName("pytest") \\
        .master("local[2]") \\
        .config("spark.sql.shuffle.partitions", "2") \\
        .getOrCreate()
    yield session
    session.stop()

# test_orders.py
from pyspark.sql import functions as F
from mypackage.transforms import filter_valid_orders, add_order_tier

def test_filter_removes_negatives(spark):
    df = spark.createDataFrame([
        ('O1', 100.0), ('O2', -50.0), ('O3', None)
    ], ['order_id', 'amount'])

    result = filter_valid_orders(df)
    assert result.count() == 1
    assert result.collect()[0]['order_id'] == 'O1'

def test_tier_boundary_conditions(spark):
    df = spark.createDataFrame([("X", 100.0), ("Y", 101.0)], ["id", "amount"])
    result = add_order_tier(df)
    rows = {r.id: r.tier for r in result.collect()}
    assert rows['X'] == 'Low'     # 100 is NOT > 100
    assert rows['Y'] == 'Medium'  # 101 IS > 100

# Run with: pytest tests/ -v
# Or: pytest tests/ -v --tb=short  (shorter tracebacks)
# Or: pytest tests/ -v -k 'orders'  (only tests matching 'orders')
""")

## Part 4: Testing Edge Cases

In [ ]:
# Test edge cases: empty DataFrames, all-null columns, schema mismatches

def test_transformation_with_all_nulls():
    all_null = spark.createDataFrame([
        ("O1", None), ("O2", None),
    ], StructType([StructField("order_id", StringType(), True),
                   StructField("amount", DoubleType(), True)]))

    result = filter_valid_orders(all_null)
    assert result.count() == 0, "All nulls should be filtered out"
    print("✓ test_transformation_with_all_nulls")


def test_enrich_with_missing_products():
    orders = spark.createDataFrame([
        ("O1", "P1", 100.0),
        ("O2", "P_MISSING", 200.0),  # product not in catalog
    ], ["order_id", "product_id", "amount"])

    products = spark.createDataFrame([
        ("P1", "Electronics"),
    ], ["product_id", "category"])

    result = enrich_with_category(orders, products)
    rows = {r.order_id: r.category for r in result.collect()}

    assert rows["O1"] == "Electronics",  "Known product should have category"
    assert rows["O2"] is None,          "Unknown product should have null category"
    print("✓ test_enrich_with_missing_products")


def test_tier_single_row():
    single = spark.createDataFrame([("O1", 999.0)], ["order_id", "amount"])
    result = add_order_tier(single)
    assert result.count() == 1
    assert result.collect()[0]["tier"] == "High"
    print("✓ test_tier_single_row")


# Test schema output
def test_add_tier_adds_column():
    df = spark.createDataFrame([("O1", 100.0)], ["order_id", "amount"])
    result = add_order_tier(df)
    assert "tier" in result.columns, "tier column must be added"
    assert dict(result.dtypes)["tier"] == "string", "tier must be StringType"
    print("✓ test_add_tier_adds_column")


test_transformation_with_all_nulls()
test_enrich_with_missing_products()
test_tier_single_row()
test_add_tier_adds_column()
print("\nAll edge case tests passed!")

## Part 5: Testing Data Quality Rules

In [ ]:
# Test the validation rules from Week 8 Topic 4

VALID_STATUSES = ["delivered", "pending", "cancelled"]

def apply_validation(df):
    return df.withColumn("_valid_amount",  F.col("amount") > 0) \
             .withColumn("_valid_status",  F.col("status").isin(VALID_STATUSES)) \
             .withColumn("_valid_customer", F.col("customer_id").isNotNull())


def test_validation_rejects_negative_amount():
    df = spark.createDataFrame([("O1", "C1", -5.0, "pending")],
                                ["order_id", "customer_id", "amount", "status"])
    result = apply_validation(df)
    row = result.collect()[0]
    assert row["_valid_amount"] == False
    print("✓ test_validation_rejects_negative_amount")


def test_validation_rejects_invalid_status():
    df = spark.createDataFrame([("O1", "C1", 100.0, "UNKNOWN")],
                                ["order_id", "customer_id", "amount", "status"])
    result = apply_validation(df)
    row = result.collect()[0]
    assert row["_valid_status"] == False
    print("✓ test_validation_rejects_invalid_status")


def test_validation_passes_good_record():
    df = spark.createDataFrame([("O1", "C1", 100.0, "delivered")],
                                ["order_id", "customer_id", "amount", "status"])
    result = apply_validation(df)
    row = result.collect()[0]
    assert row["_valid_amount"]  == True
    assert row["_valid_status"]  == True
    assert row["_valid_customer"] == True
    print("✓ test_validation_passes_good_record")


test_validation_rejects_negative_amount()
test_validation_rejects_invalid_status()
test_validation_passes_good_record()
print("\nAll validation tests passed!")

## Exercises

1. Write tests for `enrich_with_category` covering: (a) all products found, (b) some products missing (null category), (c) empty orders DataFrame, (d) empty products DataFrame.
2. Write a `generate_test_orders(spark, n, seed=42)` function that creates reproducible test DataFrames with predictable values (no rand()). Use it in 3 different tests.
3. A colleague's test passes locally but fails in CI. The test asserts `result.collect() == expected.collect()`. What is the likely cause and how do you fix it?
4. Write a test that verifies a Delta MERGE is idempotent: run it twice with the same input and assert the output row count is the same.
5. What are the tradeoffs of `scope='session'` vs `scope='function'` for the `SparkSession` pytest fixture?

In [ ]:
# Exercise 2: Reproducible test data generator
def generate_test_orders(n: int, seed: int = 42):
    """Generate reproducible test orders without random functions."""
    statuses = ["delivered", "pending", "cancelled"]
    products = ["P001", "P002", "P003"]
    customers = ["C001", "C002", "C003", "C004"]

    # Deterministic using modulo (no rand())
    rows = [
        (
            f"O{str(i).zfill(3)}",
            customers[i % len(customers)],
            products[i % len(products)],
            float((i * 37 + seed) % 1000 + 1),  # deterministic amount > 0
            statuses[i % len(statuses)],
        )
        for i in range(1, n + 1)
    ]
    return spark.createDataFrame(rows, ["order_id", "customer_id", "product_id", "amount", "status"])

# Test with reproducible data
test_data = generate_test_orders(10, seed=42)
print(f"Generated {test_data.count()} test orders:")
test_data.show(5)

# Same seed = same data
same_data = generate_test_orders(10, seed=42)
assert test_data.collect() == same_data.collect(), "Same seed must produce same data"
print("✓ Reproducibility confirmed (seed=42 always produces same output)")

# Answer Q3
print("""
Answer Q3: collect() == collect() fails in CI
Cause: Row ORDER is non-deterministic in Spark.
       Local mode may have consistent ordering by coincidence,
       but CI may use a different execution order.

Fix: Sort both before comparing
  actual_rows   = sorted(result.collect(),   key=lambda r: r['order_id'])
  expected_rows = sorted(expected.collect(), key=lambda r: r['order_id'])
  assert actual_rows == expected_rows

Answer Q5: session vs function scope
  scope='session' (recommended):
    ✓ One SparkSession for all tests → fast (no JVM restart)
    ✗ Tests share state → cached data, config changes can leak between tests
    → Fix: each test creates its own DataFrames, doesn't mutate shared state

  scope='function':
    ✓ Clean state for every test
    ✗ JVM startup per test → extremely slow (5-30s per test)
    → Only use if tests genuinely need isolated SparkContext
""")